In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Load the dataset
df = pd.read_csv('/content/income.csv')

# Display the first 5 rows and basic info to understand the data structure
display(df.head())
print(df.info())


,age,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week,income_level
0,39,77516,13,2174,0,40,0
1,50,83311,13,0,0,13,0
2,38,215646,9,0,0,40,0
3,53,234721,7,0,0,40,0
4,28,338409,13,0,0,40,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             48842 non-null  int64
 1   fnlwgt          48842 non-null  int64
 2   education_num   48842 non-null  int64
 3   capital_gain    48842 non-null  int64
 4   capital_loss    48842 non-null  int64
 5   hours_per_week  48842 non-null  int64
 6   income_level    48842 non-null  int64
dtypes: int64(7)
memory usage: 2.6 MB
None


In [5]:
# Assuming 'income_level' is the target variable
if 'Income' in df.columns:
    df = df.rename(columns={'Income': 'income_level'})

# Separate features (X) and target (y)
X = df.drop('income_level', axis=1)
y = df['income_level']

# Convert target variable to numerical (0 and 1)
# Assuming income has two classes like '<=50K' and '>50K'
# It's important to check the actual unique values in the 'income_level' column
print(f"Unique values in target variable 'income_level': {y.unique()}")

# Map the target variable to 0 and 1 if it's categorical
# Adjust this mapping based on actual unique values if necessary
if y.dtype == 'object':
    unique_income_values = y.unique()
    if len(unique_income_values) == 2:
        # Assuming the first value is the negative class (0) and the second is the positive class (1)
        y = y.map({unique_income_values[0]: 0, unique_income_values[1]: 1})
    else:
        print("Warning: 'income_level' column has more than two unique categorical values. Please inspect and adjust mapping if needed.")


# Identify categorical and numerical features in X
categorical_features = X.select_dtypes(include=['object']).columns
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns

# Apply one-hot encoding to categorical features
X = pd.get_dummies(X, columns=categorical_features, drop_first=True)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")


Unique values in target variable 'income_level': [0 1]
Shape of X_train: (34189, 6)
Shape of X_test: (14653, 6)
Shape of y_train: (34189,)
Shape of y_test: (14653,)


In [6]:
# Initialize AdaBoost Classifier with n_estimators=10
ada_clf_10 = AdaBoostClassifier(n_estimators=10, random_state=42)

# Train the model
ada_clf_10.fit(X_train, y_train)

# Make predictions on the test set
y_pred_10 = ada_clf_10.predict(X_test)

# Calculate the accuracy score
accuracy_10 = accuracy_score(y_test, y_pred_10)
print(f"AdaBoost Classifier with n_estimators=10 Accuracy: {accuracy_10:.4f}")


AdaBoost Classifier with n_estimators=10 Accuracy: 0.8277


In [7]:
best_score = 0
best_n_estimators = 0

# Try a range of n_estimators values
for n in range(1, 101, 5): # From 1 to 100, stepping by 5
    ada_clf = AdaBoostClassifier(n_estimators=n, random_state=42)
    ada_clf.fit(X_train, y_train)
    y_pred = ada_clf.predict(X_test)
    score = accuracy_score(y_test, y_pred)

    if score > best_score:
        best_score = score
        best_n_estimators = n

    print(f"n_estimators: {n}, Accuracy: {score:.4f}")

print(f"\nBest score obtained: {best_score:.4f} with n_estimators: {best_n_estimators}")


n_estimators: 1, Accuracy: 0.8007
n_estimators: 6, Accuracy: 0.8214
n_estimators: 11, Accuracy: 0.8240
n_estimators: 16, Accuracy: 0.8244
n_estimators: 21, Accuracy: 0.8248
n_estimators: 26, Accuracy: 0.8285
n_estimators: 31, Accuracy: 0.8295
n_estimators: 36, Accuracy: 0.8300
n_estimators: 41, Accuracy: 0.8301
n_estimators: 46, Accuracy: 0.8302
n_estimators: 51, Accuracy: 0.8292
n_estimators: 56, Accuracy: 0.8292
n_estimators: 61, Accuracy: 0.8297
n_estimators: 66, Accuracy: 0.8302
n_estimators: 71, Accuracy: 0.8303
n_estimators: 76, Accuracy: 0.8305
n_estimators: 81, Accuracy: 0.8307
n_estimators: 86, Accuracy: 0.8305
n_estimators: 91, Accuracy: 0.8305
n_estimators: 96, Accuracy: 0.8308

Best score obtained: 0.8308 with n_estimators: 96
